In [5]:
import requests

AGENTS = [
    "http://localhost:8000",  # math-agent
    "http://localhost:8001"   # finance-agent
]

def discover_capabilities():
    registry = []

    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()

        registry.append({
            "agent": info["name"],
            "description": info["description"],
            "endpoint": info["endpoint"],
            "tools": tools["tools"]
        })

    return registry

In [6]:
import re

def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    steps = []

    # Step 1: Interest
    if "interest" in query:
        steps.append({
            "agent": "finance-agent",
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        })

    # Step 2: Add
    if "add" in query:
        steps.append({
            "agent": "math-agent",
            "tool": "add",
            "args": {
                "b": numbers[-1]  # will inject 'a' later
            }
        })

    return steps

In [7]:
import re

def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    steps = []

    # Step 1: Interest
    if "interest" in query:
        steps.append({
            "agent": "finance-agent",
            "tool": "calculate_interest",
            "args": {
                "principal": numbers[0],
                "rate": numbers[1],
                "time": numbers[2]
            }
        })

    # Step 2: Add
    if "add" in query:
        steps.append({
            "agent": "math-agent",
            "tool": "add",
            "args": {
                "b": numbers[-1]  # will inject 'a' later
            }
        })

    return steps

In [8]:
import requests

def invoke(agent, registry, payload):
    for a in registry:
        if a["agent"] == agent:
            return requests.post(a["endpoint"], json=payload).json()

In [9]:
def execute(query):
    print("\n🔍 QUERY:", query)

    registry = discover_capabilities()

    steps = plan(query)

    print("📋 PLAN:", steps)

    result = None

    for i, step in enumerate(steps):

        # Inject previous result into next step
        if step["tool"] == "add":
            step["args"]["a"] = result

        payload = {
            "tool": step["tool"],
            "args": step["args"]
        }

        print(f"\n🚀 STEP {i+1}: {step['agent']} → {payload}")

        response = invoke(step["agent"], registry, payload)

        result = response["result"]

        print("✅ RESULT:", result)

    return result

In [10]:
query = "calculate interest for 2000 at 10 for 1 year and then add 500"

print("\n🎯 FINAL:", execute(query))


🔍 QUERY: calculate interest for 2000 at 10 for 1 year and then add 500
📋 PLAN: [{'agent': 'finance-agent', 'tool': 'calculate_interest', 'args': {'principal': 2000.0, 'rate': 10.0, 'time': 1.0}}, {'agent': 'math-agent', 'tool': 'add', 'args': {'b': 500.0}}]

🚀 STEP 1: finance-agent → {'tool': 'calculate_interest', 'args': {'principal': 2000.0, 'rate': 10.0, 'time': 1.0}}
✅ RESULT: 200.0

🚀 STEP 2: math-agent → {'tool': 'add', 'args': {'b': 500.0, 'a': 200.0}}
✅ RESULT: 700.0

🎯 FINAL: 700.0


## What Just Happened 
| Step          | Layer |
| ------------- | ----- |
| Plan          | A2A   |
| Delegate      | A2A   |
| Execute       | MCP   |
| Chain results | A2A   |

